In [1]:
import pandas as pd

data = {
    'Date': ['2016-06-30', '2017-06-30', '2018-06-30', '2019-06-30', '2020-06-30',
             '2021-06-30', '2022-06-30', '2023-06-30', '2024-06-30', '2025-06-30', '2026-05-29'],
    'Index_Value': [4586, 5961, 5733, 5138, 4067, 8398, 8088, 10493, 17367, 17642, 16992],
    'Investment':  [500000, 500000, 500000, 500000, 500000, 500000, 500000, 500000, 500000, 500000, 0]
}

df = pd.DataFrame(data)
df['Date'] = pd.to_datetime(df['Date'])

# ── Step 1: Buy units on each SIP date ──────────────────────────────────────
# Units bought = Investment / Index_Value on that day
df['Units_Bought'] = df['Investment'] / df['Index_Value']

# ── Step 2: Cumulative units held after each SIP ────────────────────────────
df['Cumulative_Units'] = df['Units_Bought'].cumsum()

# ── Step 3: Portfolio value at each row's index level ───────────────────────
df['Portfolio_Value'] = df['Cumulative_Units'] * df['Index_Value']

# ── Step 4: Total invested so far at each row ───────────────────────────────
df['Total_Invested'] = df['Investment'].cumsum()

# ── Step 5: Absolute gain and return % at each point ────────────────────────
df['Absolute_Gain'] = df['Portfolio_Value'] - df['Total_Invested']
df['Return_%']      = (df['Absolute_Gain'] / df['Total_Invested']) * 100

# ── Step 6: Year-over-year portfolio change ─────────────────────────────────
df['YoY_Change'] = df['Portfolio_Value'].diff()

# ── Print ────────────────────────────────────────────────────────────────────
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

print("=" * 110)
print("           NIFTY 250 SMALLCAP — SIP PORTFOLIO TRACKER  (₹5,00,000 / year on 1st July)")
print("=" * 110)

header = f"{'Date':<14} {'Index':>8} {'Invested(₹)':>13} {'Units Bought':>13} {'Total Units':>12} " \
         f"{'Ttl Invested(₹)':>16} {'Portfolio(₹)':>14} {'Gain(₹)':>13} {'Return%':>9}"
print(header)
print("-" * 110)

for _, row in df.iterrows():
    tag = "  ← VALUATION" if row['Investment'] == 0 else ""
    print(
        f"{str(row['Date'].date()):<14} "
        f"{row['Index_Value']:>8,.0f} "
        f"{row['Investment']:>13,.0f} "
        f"{row['Units_Bought']:>13.4f} "
        f"{row['Cumulative_Units']:>12.4f} "
        f"{row['Total_Invested']:>16,.0f} "
        f"{row['Portfolio_Value']:>14,.0f} "
        f"{row['Absolute_Gain']:>13,.0f} "
        f"{row['Return_%']:>8.2f}%"
        f"{tag}"
    )

print("=" * 110)

# ── Final Summary ────────────────────────────────────────────────────────────
final        = df.iloc[-1]
total_inv    = final['Total_Invested']
final_value  = final['Portfolio_Value']
total_gain   = final['Absolute_Gain']
total_ret    = final['Return_%']

# XIRR
from scipy.optimize import brentq

cashflows = []
for _, row in df.iterrows():
    if row['Investment'] > 0:
        cashflows.append((row['Date'], -row['Investment']))
cashflows.append((df.iloc[-1]['Date'], final_value))

t0 = cashflows[0][0]
def npv(rate):
    return sum(amt / (1 + rate) ** ((d - t0).days / 365.0) for d, amt in cashflows)

try:
    xirr = brentq(npv, -0.999, 10.0) * 100
except Exception:
    xirr = float('nan')

print(f"\n{'FINAL SUMMARY':^110}")
print("=" * 110)
print(f"  {'Number of SIPs':<35}: 10")
print(f"  {'Total Amount Invested':<35}: ₹{total_inv:>15,.0f}")
print(f"  {'Final Portfolio Value (29 May 2026)':<35}: ₹{final_value:>15,.0f}")
print(f"  {'Total Absolute Gain':<35}: ₹{total_gain:>15,.0f}")
print(f"  {'Overall Return':<35}:  {total_ret:>14.2f}%")
print(f"  {'XIRR (Annualised Return)':<35}:  {xirr:>14.2f}%")
print("=" * 110)

           NIFTY 250 SMALLCAP — SIP PORTFOLIO TRACKER  (₹5,00,000 / year on 1st July)
Date              Index   Invested(₹)  Units Bought  Total Units  Ttl Invested(₹)   Portfolio(₹)       Gain(₹)   Return%
--------------------------------------------------------------------------------------------------------------
2016-06-30        4,586       500,000      109.0275     109.0275          500,000        500,000             0     0.00%
2017-06-30        5,961       500,000       83.8785     192.9060        1,000,000      1,149,913       149,913    14.99%
2018-06-30        5,733       500,000       87.2144     280.1204        1,500,000      1,605,930       105,930     7.06%
2019-06-30        5,138       500,000       97.3141     377.4345        2,000,000      1,939,259       -60,741    -3.04%
2020-06-30        4,067       500,000      122.9407     500.3753        2,500,000      2,035,026      -464,974   -18.60%
2021-06-30        8,398       500,000       59.5380     559.9132        3,000